# Trabajo Práctico de Laboratorio Nº 6: TBM y CBM
## Datos del grupo

* **Curso:** R4052
* **Nº de Grupo:** 2
* **Integrantes:**
  1.   Paul Leandro Encinas Claros
  2.   Franco Ezequiel Ferrario
  3.   Matías Sebastián Marchesi
  4.   Bruno Quiroga

  


## 1. Descripción del Ensayo y Objetivos
Se solicita la calibración de tres resistencias incógnitas ($R_x$) de órdenes de magnitud de $10\,\Omega$, $10\,\text{k}\Omega$ y $1\,\text{M}\Omega$ tomando como referencia tres resistencias patrones ($R_p$) previamente calibradas. 

El método implementado es el **método de comparación por técnica de deflexión**. El circuito consiste en un divisor resistivo simple alimentado por una fuente de tensión continua regulable ($V_F$). Un multímetro digital de 3 ½ dígitos es utilizado de forma alternativa en paralelo para registrar las caídas de potencial sobre el resistor patrón ($V_{Rp}$) y sobre el resistor incógnita ($V_{Rx}$). A partir de estos valores experimentales, se determina el valor corregido de las resistencias y su respectiva incertidumbre expandida asociada.

## 2. Análisis Teórico

### 2.1. Ecuación de Medición
Dado que la corriente circulante por ambas resistencias en serie es la misma (asumiendo despreciable la corriente derivada por el voltímetro digital de alta impedancia), la relación de tensiones responde directamente a la ley de Ohm:

$$R_x = R_p \cdot \frac{V_{Rx}}{V_{Rp}}$$

Donde:
* $R_x$: Resistencia incógnita bajo calibración.
* $R_p$: Resistencia patrón (con su respectivo valor e incertidumbre del TP anterior).
* $V_{Rx}$: Caída de tensión media medida sobre la resistencia incógnita.
* $V_{Rp}$: Caída de tensión media medida sobre la resistencia patrón.

### 2.2. Análisis de Incertidumbre
Dado que el modelo matemático del mensurando está compuesto puramente por operaciones de multiplicación y división de magnitudes físicas no correlacionadas (ρ = 0), resulta óptimo y equivalente propagar la incertidumbre a través de sus **componentes estándar relativas** ($u_r$). 

Aplicando la ley de propagación, la incertidumbre estándar combinada relativa de la resistencia incógnita \(u_{r}(R_x)\) se define como:

$$\left( \frac{u_c(R_x)}{R_x} \right)^2 = \left( \frac{u_c(R_p)}{R_p} \right)^2 + \left( \frac{u_c(V_{Rx})}{V_{Rx}} \right)^2 + \left( \frac{u_c(V_{Rp})}{V_{Rp}} \right)^2$$

Donde cada término representa la varianza relativa de las variables de entrada:
1. $u_r(R_p) = \frac{u_c(R_p)}{{R_p}}$ es la incertidumbre relativa de la calibración del patrón previa.
2. $u_r(V_{Rx}) = \frac{u_c(V_{Rx})}{V_{Rx}}$ es la incertidumbre relativa combinada de la tensión medida en la incógnita.
3. $u_r(V_{Rp}) = \frac{u_c(V_{Rp})}{V_{Rp}}$ es la incertidumbre relativa combinada de la tensión medida en el patrón.


In [12]:
import numpy as np
import math
import IPython
from IPython.display import display, Math
import pandas as pd
from typing import Any

def incert_A(mediciones):
    s = np.std(mediciones, ddof=1)  # ddof=1 -> grados de libertad v = n-1
    return s / math.sqrt(len(mediciones))

def incert_B_ConMedia(err_rel, nd, ctas, media):
    """
    Calcula la incertidumbre Tipo B basándose en el error relativo del manual (%)
    y la cantidad de dígitos de error (nd) sobre las cuentas leídas (ctas).
    """
    return (err_rel / 100 + nd / ctas) * media / math.sqrt(3)

def incert_Total(uA, uB):
    return math.sqrt(uA ** 2 + uB ** 2)

def incert_Comb_Rel(uX, uY, correlacion=0):
    return math.sqrt((uX['u_C']/uX['media'])**2 + (uY['u_C']/uY['media'])**2 - (2*(uX['u_C']/uX['media'])*(uY['u_C']/uY['media'])*correlacion))



# Datos de los ensayos:


# --- Ensayo 1: Rango ~10 Ohm ---
Rp1_val = 10.1      # Valor medido de la resistencia patrón en el TP anterior
u_Rp1_val = 0.05    # Incertidumbre estándar absoluta asignada a Rp1
v_patron1 = np.array([0.6202, 0.6212, 0.6243, 0.625, 0.6253, 0.6256, 0.6254, 0.6249, 0.6256, 0.6131])
v_buscada1 = np.array([0.5175, 0.5185, 0.5187, 0.519, 0.5192, 0.519, 0.5189, 0.5192, 0.5192, 0.5193])

# --- Ensayo 2: Rango ~10 kOhm ---
Rp2_val = 10000.0   # Valor nominal/calibrado de la resistencia patrón
u_Rp2_val = 5.0     # Incertidumbre estándar absoluta asignada a Rp2
v_patron2 = np.array([5.301, 5.3, 5.3, 5.3, 5.301, 5.299, 5.299, 5.3, 5.299, 5.301])
v_buscada2 = np.array([4.741, 4.741, 4.74, 4.741, 4.74, 4.741, 4.74, 4.741, 4.74, 4.741])

# --- Ensayo 3: Rango ~1 MOhm ---
Rp3_val = 1000000.0 # Valor nominal/calibrado de la resistencia patrón
u_Rp3_val = 1000.0  # Incertidumbre estándar absoluta asignada a Rp3
v_patron3 = np.array([4.383, 4.384, 4.382, 4.381, 4.38, 4.379, 4.38, 4.379, 4.38, 4.378])
v_buscada3 = np.array([5.202, 5.201, 5.202, 5.201, 5.201, 5.201, 5.201, 5.201, 5.202, 5.201])

# Factor de cobertura final para K=2
k_exp = 2


# Procesamiento de los ensayos

def procesar_ensayo(v_p, v_x, Rp_val, u_Rp_val, err_rel_tester, ctas_p, ctas_x):
    # 1. Medias de las tensiones
    media_vp = np.mean(v_p)
    media_vx = np.mean(v_x)
    
    # 2. Magnitud de salida (Mensurando Rx)
    Rx_val = Rp_val * (media_vx / media_vp)
    
    # 3. Incertidumbres de la tensión en el Patrón (V_Rp)
    uA_vp = incert_A(v_p)
    # Se asumen 2 dígitos de error sobre las cuentas leídas del display
    uB_vp = incert_B_ConMedia(err_rel=err_rel_tester, nd=2, ctas=ctas_p, media=media_vp)
    uC_vp = incert_Total(uA_vp, uB_vp)
    dict_vp = {'media': media_vp, 'u_C': uC_vp}
    
    # 4. Incertidumbres de la tensión en la Incógnita (V_Rx)
    uA_vx = incert_A(v_x)
    uB_vx = incert_B_ConMedia(err_rel=err_rel_tester, nd=2, ctas=ctas_x, media=media_vx)
    uC_vx = incert_Total(uA_vx, uB_vx)
    dict_vx = {'media': media_vx, 'u_C': uC_vx}
    
    # 5. Estructura para la propia Resistencia Patrón (Rp)
    dict_Rp = {'media': Rp_val, 'u_C': u_Rp_val}
    
    # 6. Combinación de Incertidumbres utilizando tus funciones relativas
    # Primero combinamos los dos efectos de las tensiones (V_Rx y V_Rp) sin correlación (0)
    u_comb_tensiones_rel = incert_Comb_Rel(dict_vx, dict_vp, correlacion=0)
    
    # Creamos un diccionario temporal para la componente resultante de las tensiones
    dict_tensiones_rel = {'media': 1.0, 'u_C': u_comb_tensiones_rel} # Al ser relativas, la media de control es 1.0
    dict_Rp_rel = {'media': 1.0, 'u_C': u_Rp_val / Rp_val}
    
    # Combinamos el efecto acumulado de las tensiones con la incertidumbre relativa de Rp
    u_c_Rx_rel = incert_Comb_Rel(dict_tensiones_rel, dict_Rp_rel, correlacion=0)
    
    # Pasamos la incertidumbre combinada de relativa a absoluta
    u_c_Rx = u_c_Rx_rel * Rx_val
    
    # Incertidumbre expandida
    U_Rx = k_exp * u_c_Rx
    
    return Rx_val, U_Rx, uA_vx, uB_vx, uA_vp, uB_vp

# EJECUCIÓN Y PRESENTACIÓN DE RESULTADOS

# Especificación típica del UT60E para tensión CC: 0.8%
error_instrumento = 0.8 

# Para el cálculo de cuentas (ctas), un valor típico representativo de las lecturas:
# Ensayo 1 (~10 Ohm): Lecturas en torno a 0.6V y 0.5V en escala de 4V -> 600 y 500 cuentas
r1, u1, ua_x1, ub_x1, ua_p1, ub_p1 = procesar_ensayo(v_patron1, v_buscada1, Rp1_val, u_Rp1_val, error_instrumento, 620, 518)

# Ensayo 2 (~10 kOhm): Lecturas en torno a 5.3V y 4.7V en escala de 40V -> 530 y 470 cuentas
r2, u2, ua_x2, ub_x2, ua_p2, ub_p2 = procesar_ensayo(v_patron2, v_buscada2, Rp2_val, u_Rp2_val, error_instrumento, 530, 474)

# Ensayo 3 (~1 MOhm): Lecturas en torno a 4.3V y 5.2V en escala de 40V -> 430 y 520 cuentas
r3, u3, ua_x3, ub_x3, ua_p3, ub_p3 = procesar_ensayo(v_patron3, v_buscada3, Rp3_val, u_Rp3_val, error_instrumento, 438, 520)

# Estructuración final en Dataframe de Pandas
df_resultados = pd.DataFrame({
    'Ensayo': ['Resistencia 10 Ω', 'Resistencia 10 kΩ', 'Resistencia 1 MΩ'],
    'Rx [Ω]': [r1, r2, r3],
    'U(Rx) [Ω]': [u1, u2, u3],
    'uA_Vx [V]': [ua_x1, ua_x2, ua_x3],
    'uB_Vx [V]': [ub_x1, ub_x2, ub_x3],
    'uA_Vp [V]': [ua_p1, ua_p2, ua_p3],
    'uB_Vp [V]': [ub_p1, ub_p2, ub_p3]
})

# Formateo
print("\n--- RESULTADOS FINALES DE CALIBRACIÓN ---")
display(df_resultados.round(4))

print("\n--- EXPRESIÓN DE LOS MENSURANDOS ---")
display(Math(f"R_{{x1}} = ({r1:.2f} \\pm {u1:.2f})\\,\\Omega \\quad [k=2]"))
display(Math(f"R_{{x2}} = ({r2/1000:.2f} \\pm {u2/1000:.2f})\\,\\text{{k}}\\Omega \\quad [k=2]"))
display(Math(f"R_{{x3}} = ({r3/1000000:.3f} \\pm {u3/1000000:.3f})\\,\\text{{M}}\\Omega \\quad [k=2]"))



--- RESULTADOS FINALES DE CALIBRACIÓN ---


,Ensayo,Rx [Ω],U(Rx) [Ω],uA_Vx [V],uB_Vx [V],uA_Vp [V],uB_Vp [V]
0,Resistencia 10 Ω,8.410700e+00,0.1824,0.0002,0.0036,0.0013,0.0040
1,Resistencia 10 kΩ,8.944528e+03,175.4867,0.0002,0.0334,0.0003,0.0360
2,Resistencia 1 MΩ,1.187349e+06,23798.3661,0.0002,0.0356,0.0006,0.0318



--- EXPRESIÓN DE LOS MENSURANDOS ---


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4. Conclusiones y Discusión de Resultados

Analizando los valores obtenidos, podemos realizar las siguientes observaciones:

1. **Predominancia de la Incertidumbre Tipo B:** Al comparar el orden de magnitud de la componente estadística (Tipo A) con la del instrumental (Tipo B), se observa que la **incertidumbre Tipo B es significativamente mayor** ($u_B \gg u_A$). Esto demuestra que las fluctuaciones ambientales y el ruido aleatorio del laboratorio estuvieron controlados, y que el límite de precisión del ensayo quedó impuesto por la tolerancia de fábrica y resolución de los multímetros digitales utilizados.
2. **Efecto de Carga del Voltímetro en Alta Impedancia (Ensayo de 1 M$\Omega$):** En el tercer ensayo, la resistencia incógnita y el patrón son del orden de los $1\,\text{M}\Omega$. Como la resistencia interna del multímetro UT60E se encuentra en torno a los $10\,\text{M}\Omega$ en modo voltímetro, su conexión en paralelo altera sustancialmente la corriente del circuito serie original. Esto introduce un **error sistemático por efecto de carga** que reduce la tensión medida en los resistores. Para una calibración formal de alta exactitud en este rango, el método por deflexión directa debe ser sustituido por métodos otros métodos como el Puente de Wheatstone.
3. **Validación del Método:** El método de comparación por técnica de deflexión demostró ser robusto, rápido y de implementación simple para valores de resistencia intermedios ($10\,\Omega$ a $10\,\text{k}\Omega$), donde la impedancia del instrumento de medición no afecta la dinámica del circuito serie. Los resultados finales quedan debidamente expresados con su intervalo de confianza asociado.
